# 03 — Data Preprocessing

Build a reusable train/test pipeline: harmonise labels, impute, engineer a few features, encode, scale (**fit on train only**), and persist artefacts for baselines, GA, PSO, and TabNet.

In [1]:
from pathlib import Path
import random

import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "environment.yml").exists()),
    cwd,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for d in (DATA_INTERIM, DATA_PROCESSED, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Random seed  : {RANDOM_SEED}")

Project root : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction
Random seed  : 42


In [2]:
import json
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin

df = pd.read_csv(DATA_INTERIM / "ecomm_validated.csv")
print(df.shape)

(5630, 20)


## Why these choices?

| Step | Choice | Justification |
|------|--------|----------------|
| Harmonise categories | Map Phone→Mobile Phone, CC→Credit Card, COD→Cash on Delivery, Mobile→Mobile Phone | EDA found overlapping labels that would create redundant one-hot columns |
| Impute | Median (numeric), most_frequent (categorical) | ~4–5% missingness; median resists skew |
| Ordinals | Keep CityTier, SatisfactionScore, Complain as integers | Preserve order; avoid exploding cardinality |
| Scale | RobustScaler on continuous numerics | Skew + IQR outliers from EDA |
| Split | Stratified 80/20, seed 42 | Preserve churn rate; reproducibility |

In [3]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Light domain features justified by EDA."""

    def fit(self, X, y=None):
        self.recency_threshold_ = float(X["DaySinceLastOrder"].median())
        return self

    def transform(self, X):
        X = X.copy()
        X["PreferredLoginDevice"] = X["PreferredLoginDevice"].replace({"Phone": "Mobile Phone"})
        X["PreferredPaymentMode"] = X["PreferredPaymentMode"].replace({
            "CC": "Credit Card", "COD": "Cash on Delivery",
        })
        X["PreferedOrderCat"] = X["PreferedOrderCat"].replace({"Mobile": "Mobile Phone"})

        tenure = X["Tenure"].fillna(X["Tenure"].median() if X["Tenure"].notna().any() else 0)
        X["TenureBin"] = pd.cut(
            tenure, bins=[-0.1, 3, 9, 15, 1e9],
            labels=[0, 1, 2, 3],
        ).astype(float)

        X["IsDormant"] = (X["DaySinceLastOrder"] > self.recency_threshold_).astype(float)
        X["UnhappyComplain"] = (
            (X["Complain"].fillna(0) == 1) & (X["SatisfactionScore"].fillna(3) <= 2)
        ).astype(float)
        return X


fe = FeatureEngineer()
df_fe = fe.fit_transform(df)
print("Recency threshold (global preview — train refit below):", fe.recency_threshold_)
df_fe[["TenureBin", "IsDormant", "UnhappyComplain"]].head()

Recency threshold (global preview — train refit below): 3.0


,TenureBin,IsDormant,UnhappyComplain
0,1.0,1.0,1.0
1,1.0,0.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,0.0
4,0.0,0.0,0.0


In [4]:
TARGET = "Churn"
ID_COL = "CustomerID"

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y,
)
print("Train", X_train.shape, "churn", y_train.mean())
print("Test ", X_test.shape, "churn", y_test.mean())

# Fit FE on train only
fe = FeatureEngineer()
X_train_fe = fe.fit_transform(X_train)
X_test_fe = fe.transform(X_test)

numeric_continuous = [
    "Tenure", "WarehouseToHome", "HourSpendOnApp", "NumberOfDeviceRegistered",
    "NumberOfAddress", "OrderAmountHikeFromlastYear", "CouponUsed", "OrderCount",
    "DaySinceLastOrder", "CashbackAmount",
]
ordinal_as_is = ["CityTier", "SatisfactionScore", "Complain", "TenureBin", "IsDormant", "UnhappyComplain"]
categorical = ["PreferredLoginDevice", "PreferredPaymentMode", "Gender", "PreferedOrderCat", "MaritalStatus"]

# Drop ID from modelling matrix
for part in (X_train_fe, X_test_fe):
    assert ID_COL in part.columns

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ]), numeric_continuous),
        ("ord", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), ordinal_as_is),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
    ],
    remainder="drop",
)

X_train_model = X_train_fe.drop(columns=[ID_COL])
X_test_model = X_test_fe.drop(columns=[ID_COL])

X_train_p = preprocess.fit_transform(X_train_model)
X_test_p = preprocess.transform(X_test_model)

# Feature names
num_names = numeric_continuous
ord_names = ordinal_as_is
cat_names = list(preprocess.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical))
feature_names = num_names + ord_names + cat_names
print("n_features:", len(feature_names))
print(feature_names[:15], "...")

Train (4504, 19) churn 0.16829484902309058
Test  (1126, 19) churn 0.16873889875666073
n_features: 33
['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'NumberOfAddress', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount', 'CityTier', 'SatisfactionScore', 'Complain', 'TenureBin', 'IsDormant'] ...


In [5]:
bundle = {
    "feature_engineer": fe,
    "preprocessor": preprocess,
    "feature_names": feature_names,
    "numeric_continuous": numeric_continuous,
    "ordinal_as_is": ordinal_as_is,
    "categorical": categorical,
    "id_col": ID_COL,
    "random_seed": RANDOM_SEED,
}
joblib.dump(bundle, MODELS_DIR / "preprocessor.joblib")

# Persist matrices + IDs for later notebooks
pd.DataFrame(X_train_p, columns=feature_names).to_csv(DATA_PROCESSED / "X_train.csv", index=False)
pd.DataFrame(X_test_p, columns=feature_names).to_csv(DATA_PROCESSED / "X_test.csv", index=False)
y_train.to_csv(DATA_PROCESSED / "y_train.csv", index=False)
y_test.to_csv(DATA_PROCESSED / "y_test.csv", index=False)
X_train[[ID_COL]].to_csv(DATA_PROCESSED / "id_train.csv", index=False)
X_test[[ID_COL]].to_csv(DATA_PROCESSED / "id_test.csv", index=False)

meta = {
    "n_train": int(len(y_train)),
    "n_test": int(len(y_test)),
    "n_features": len(feature_names),
    "train_churn_rate": float(y_train.mean()),
    "test_churn_rate": float(y_test.mean()),
    "recency_threshold": float(fe.recency_threshold_),
}
(DATA_PROCESSED / "preprocess_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(meta)

{'n_train': 4504, 'n_test': 1126, 'n_features': 33, 'train_churn_rate': 0.16829484902309058, 'test_churn_rate': 0.16873889875666073, 'recency_threshold': 3.0}


## Leakage checklist

- [x] Target `Churn` never used inside transformers
- [x] Imputer / scaler / one-hot / recency threshold fitted on **train only**
- [x] Stratified split before fitting
- [x] `CustomerID` excluded from model matrix
- [x] Test transform uses train statistics only

**Next:** Notebook `04` — Logistic Regression + small Random Forest baselines.